# [STARTER] Udaplay Project

## Part 01 - Offline RAG

In this part of the project, you'll build your VectorDB using Chroma.

The data is inside folder `project/starter/games`. Each file will become a document in the collection you'll create.
Example.:
```json
{
  "Name": "Gran Turismo",
  "Platform": "PlayStation 1",
  "Genre": "Racing",
  "Publisher": "Sony Computer Entertainment",
  "Description": "A realistic racing simulator featuring a wide array of cars and tracks, setting a new standard for the genre.",
  "YearOfRelease": 1997
}
```


### Setup

In [1]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [2]:
import os
import json
import chromadb
from chromadb.utils import embedding_functions
from dotenv import load_dotenv

In [3]:
# TODO: Create a .env file with the following variables
# OPENAI_API_KEY="YOUR_KEY"
# CHROMA_OPENAI_API_KEY="YOUR_KEY"
# TAVILY_API_KEY="YOUR_KEY"

In [4]:
load_dotenv()

True

### VectorDB Instance

In [5]:
chroma_client = chromadb.PersistentClient(path="chromadb")

### Collection

In [6]:
embedding_fn = embedding_functions.OpenAIEmbeddingFunction(
    api_key=os.getenv("OPENAI_API_KEY"),
    model_name="text-embedding-3-small"
)

In [7]:
collection = chroma_client.get_or_create_collection(
    name="udaplay",
    embedding_function=embedding_fn
)

### Add documents

In [8]:
# Make sure you have a directory "project/starter/games"
data_dir = "games"

for file_name in sorted(os.listdir(data_dir)):
    if not file_name.endswith(".json"):
        continue

    file_path = os.path.join(data_dir, file_name)
    with open(file_path, "r", encoding="utf-8") as f:
        game = json.load(f)

    # You can change what text you want to index
    content = f"[{game['Platform']}] {game['Name']} ({game['YearOfRelease']}) - {game['Description']}"

    # Use file name (like 001) as ID
    doc_id = os.path.splitext(file_name)[0]

    collection.add(
        ids=[doc_id],
        documents=[content],
        metadatas=[game]
    )

### Semantic Search

In [9]:
query = "realistic racing games with lots of cars"
n_results = 3

results = collection.query(
    query_texts=[query],
    n_results=n_results
)

print(f"Top {n_results} results for: '{query}'\n")
for i, (doc, meta, distance) in enumerate(zip(
    results["documents"][0],
    results["metadatas"][0],
    results["distances"][0],
)):
    print(f"Result {i + 1} (distance: {distance:.4f})")
    print(f"  Name    : {meta['Name']}")
    print(f"  Platform: {meta['Platform']}")
    print(f"  Genre   : {meta['Genre']}")
    print(f"  Year    : {meta['YearOfRelease']}")
    print(f"  Content : {doc}")
    print()

Top 3 results for: 'realistic racing games with lots of cars'

Result 1 (distance: 0.3930)
  Name    : Gran Turismo
  Platform: PlayStation 1
  Genre   : Racing
  Year    : 1997
  Content : [PlayStation 1] Gran Turismo (1997) - A realistic racing simulator featuring a wide array of cars and tracks, setting a new standard for the genre.

Result 2 (distance: 0.4341)
  Name    : Gran Turismo 5
  Platform: PlayStation 3
  Genre   : Racing
  Year    : 2010
  Content : [PlayStation 3] Gran Turismo 5 (2010) - A comprehensive racing simulator featuring a vast selection of vehicles and tracks, with realistic driving physics.

Result 3 (distance: 0.6377)
  Name    : Grand Theft Auto: San Andreas
  Platform: PlayStation 2
  Genre   : Action-adventure
  Year    : 2004
  Content : [PlayStation 2] Grand Theft Auto: San Andreas (2004) - An expansive open-world game set in the fictional state of San Andreas, following the story of Carl 'CJ' Johnson.

